In [0]:
#pip install optuna

In [0]:
import optuna

def tool_hyperparameter_tuning_and_train(data_dict, n_trials=5):
    """Executa a otimização de hiperparâmetros com Optuna e loga no MLflow."""
    X_train, X_test, y_train, y_test = data_dict["X_train"], data_dict["X_test"], data_dict["Y_train"] if "Y_train" in data_dict else data_dict["y_train"], data_dict["y_test"]
    
    mlflow.set_experiment("/Shared/emissor_a_behavior_score")
    
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "random_state": 42
        }
        
        with mlflow.start_run(nested=True):
            model = xgb.XGBClassifier(**params)
            model.fit(X_train, y_train)
            
            preds = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, preds)
            
            mlflow.log_params(params)
            mlflow.log_metric("val_auc", auc)
            
        return auc

    print("[Agente] Iniciando tunning de hiperparâmetros com Optuna...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    best_params = study.best_params
    print(f"[Agente] Melhores parâmetros encontrados: {best_params}")
    
    # Treina o modelo final com os melhores parâmetros
    best_model = xgb.XGBClassifier(**best_params, random_state=42)
    best_model.fit(X_train, y_train)
    
    return {"model": best_model, "best_params": best_params, "best_auc": study.best_value}